# Natural Language Processing with Disaster Tweets

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import time
import os
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

In [ ]:
url_train='/kaggle/input/nlp-getting-started/train.csv'
url_test='/kaggle/input/nlp-getting-started/test.csv'
url_submission='/kaggle/input/nlp-getting-started/sample_submission.csv'
train=pd.read_csv(url_train,sep=',')
test=pd.read_csv(url_test,sep=',')
submission=pd.read_csv(url_submission,sep=',')

In [ ]:
plt.figure(figsize=(10,8))
sns.countplot(data=train,x='target')
plt.title('Target')
plt.show()

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Descargar lo mínimo (¡esto sí funciona!)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def limpiar_texto_simple(texto):
    texto = texto.lower()
    texto = re.sub(r"http\S+|www\S+|https\S+", '', texto)
    texto = re.sub(r'@\w+|#\w+', '', texto)
    texto = re.sub(r'[^a-z\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    palabras = word_tokenize(texto)
    palabras = [w for w in palabras if w not in stop_words]
    palabras = [lemmatizer.lemmatize(w) for w in palabras]  # <- SIN pos_tag
    
    return ' '.join(palabras)


In [ ]:
train['text_clean'] = train['text'].astype(str).apply(limpiar_texto_simple)
train

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizar texto
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train['text_clean'])

X_seq = tokenizer.texts_to_sequences(train['text_clean'])
X_pad = pad_sequences(X_seq, maxlen=100)  # Cortar/padding a 100 palabras

# Etiquetas
y = train['target']


# Parámetros clave
max_words = 10000     # Vocabulario máximo
max_len = 100         # Longitud máxima por texto
embedding_dim = 64    # Dimensión del vector de palabras

# Tokenizar
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train['text_clean'])

X_seq = tokenizer.texts_to_sequences(train['text_clean'])
X_pad = pad_sequences(X_seq, maxlen=max_len)
y = train['target']

# Dividir en train y val
X_train, X_val, y_train, y_val = train_test_split(X_pad, y, test_size=0.2, random_state=42)

# Modelo mejorado con LSTM
model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Compilar
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Callback
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Entrenar
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_data=(X_val, y_val),
                    callbacks=[early_stop])


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predecir en el set de validación
y_pred_proba = model.predict(X_val)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()  # Convertir a 0/1

# Reporte de clasificación
print(classification_report(y_val, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
print("Matriz de confusión:\n", cm)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1], yticklabels=[0,1])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label='AUC = {:.2f}'.format(roc_auc))
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Gráfica de pérdida
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Gráfica de precisión
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## **TEST**

In [ ]:
test['text_clean'] = test['text'].astype(str).apply(limpiar_texto_simple)
test

In [ ]:
# Convertir texto limpio en secuencias
X_test_seq = tokenizer.texts_to_sequences(test['text_clean'])

# Rellenar secuencias
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)


In [ ]:
# Predecir probabilidades
y_test_proba = model.predict(X_test_pad)

# Convertir a etiquetas binarias (0 o 1)
y_test_pred = (y_test_proba > 0.5).astype(int).flatten()


In [ ]:
submission['target']=y_test_pred
# Guardar archivo
submission.to_csv('submission.csv', index=False)

## **Modelo Mejorado**

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import MaxPooling1D,Bidirectional
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
# Balancear Clases
class_weights = compute_class_weight('balanced', classes=[0,1], y=y_train)
class_weights = dict(enumerate(class_weights))

# Ajustar el tokenizer y padding
tokenizer = Tokenizer(num_words=15000)  # Más palabras
X_seq = tokenizer.texts_to_sequences(train['text_clean'])
X_pad = pad_sequences(X_seq, maxlen=150)  # Mayor longitud

# Optimizador con decaimiento
optimizer = Adam(learning_rate=0.0001, decay=1e-4)

# Callbacks adicionales
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)


model = Sequential([
    Embedding(15000, 128, input_length=150),
    
    Conv1D(128, 5, activation='relu', padding='same'),
    MaxPooling1D(2),
    Dropout(0.3),
    
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dropout(0.4),
    
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy', 
                       tf.keras.metrics.Precision(),
                       tf.keras.metrics.Recall()])


history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=128,  # Mayor batch size
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler],
    class_weight=class_weights  # Si hay desbalance de clases
)

In [ ]:
# Gráfica de pérdida
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Gráfica de precisión
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Predecir en el set de validación
y_pred_proba = model.predict(X_val)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()  # Convertir a 0/1

# Reporte de clasificación
print(classification_report(y_val, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
print("Matriz de confusión:\n", cm)

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1], yticklabels=[0,1])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()


In [ ]:

fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label='AUC = {:.2f}'.format(roc_auc))
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Predecir probabilidades
y_test_proba = model.predict(X_test_pad)

# Convertir a etiquetas binarias (0 o 1)
y_test_pred = (y_test_proba > 0.5).astype(int).flatten()

In [ ]:
submission['target']=y_test_pred
# Guardar archivo
submission.to_csv('submission.csv', index=False)